# 🩺 Breast Cancer Detection Using AI & Deep Learning

**Datasets:** CBIS-DDSM · Breast Histopathology Images  
**Pipeline:** Enhancement → Segmentation → CNN Classification  
**Models:** 3 CNNs (Histopathology, Watershed, Canny)

---

## 0. Setup & Imports

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf

sys.path.insert(0, os.path.abspath('..'))
import config as cfg
from utils import *

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU Available      : {tf.config.list_physical_devices("GPU")}')
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Synthetic Data Preview (Demo Mode)

In [ ]:
# Generate a synthetic mammogram for pipeline demonstration
synthetic_mammo = generate_synthetic_mammogram()
print(f'Synthetic mammogram shape: {synthetic_mammo.shape}, dtype: {synthetic_mammo.dtype}')

plt.figure(figsize=(5,5), facecolor='#0d1117')
plt.imshow(synthetic_mammo, cmap='gray')
plt.title('Synthetic Mammogram', color='white')
plt.axis('off')
plt.show()

## 2. Image Enhancement Pipeline

In [ ]:
enhanced = enhance_mammogram(synthetic_mammo)
plot_enhancement_pipeline(enhanced,
    save_path=os.path.join(cfg.PLOTS_DIR, 'nb_enhancement.png'))

In [ ]:
# Pixel intensity histograms across enhancement stages
plot_intensity_histogram(enhanced,
    save_path=os.path.join(cfg.PLOTS_DIR, 'nb_histograms.png'))

## 3. Image Segmentation

In [ ]:
clahe_img = enhanced['clahe']

# Watershed
ws_results = apply_watershed(clahe_img)
plot_watershed_results(ws_results,
    save_path=os.path.join(cfg.PLOTS_DIR, 'nb_watershed.png'))

In [ ]:
# Canny
cn_results = apply_canny(clahe_img)
plot_canny_results(cn_results,
    save_path=os.path.join(cfg.PLOTS_DIR, 'nb_canny.png'))

In [ ]:
# Side-by-side comparison
plot_segmentation_comparison(clahe_img, ws_results, cn_results,
    save_path=os.path.join(cfg.PLOTS_DIR, 'nb_seg_comparison.png'))

## 4. Histopathology Dataset Analysis

In [ ]:
# Generate synthetic patches
images, labels = generate_synthetic_histopathology(n_samples=1000)
print(f'Dataset shape : {images.shape}')
print(f'Label balance : {np.bincount(labels)}')

plot_class_distribution(labels, cfg.HISTO_CLASSES,
    'Histopathology Dataset (Demo)',
    save_path=os.path.join(cfg.PLOTS_DIR, 'nb_histo_dist.png'))

In [ ]:
# Sample patches
fig, axes = plt.subplots(2, 8, figsize=(18, 5))
fig.patch.set_facecolor('#0d1117')
for cls, row in enumerate(axes):
    cls_imgs = images[labels == cls][:8]
    for ax, img in zip(row, cls_imgs):
        ax.imshow(img)
        ax.axis('off')
        ax.set_facecolor('#161b22')
    row[0].set_ylabel(cfg.HISTO_CLASSES[cls], color='white', fontsize=9, rotation=90)
fig.suptitle('Sample Histopathology Patches', color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. CNN Model Architecture

In [ ]:
histo_model = build_histopathology_cnn()
print_model_summary(histo_model)

In [ ]:
mammo_model = build_mammogram_cnn()
print_model_summary(mammo_model)

## 6. Train Models (Demo)

In [ ]:
# Run quick demo training (5 epochs)
!python ../train.py --mode demo --epochs 5 --batch-size 16

## 7. Load Results & Visualize

In [ ]:
# Display saved plots
plots_to_show = [
    ('histo_history.png',   'Training History'),
    ('histo_confusion.png', 'Confusion Matrix'),
    ('accuracy_summary.png','Accuracy Comparison'),
]

for fname, title in plots_to_show:
    fpath = os.path.join(cfg.PLOTS_DIR, fname)
    if os.path.exists(fpath):
        img = plt.imread(fpath)
        plt.figure(figsize=(12, 5), facecolor='#0d1117')
        plt.imshow(img)
        plt.title(title, color='white')
        plt.axis('off')
        plt.show()
    else:
        print(f'[Not found] {fname} — run training first')

## 8. Single Image Prediction

Replace `demo=True` with your own image path after training.

In [ ]:
!python ../predict.py --demo --type histo

In [ ]:
!python ../predict.py --demo --type mammogram

---
## Summary

| Model | Input | Reported Accuracy |
|---|---|---|
| Histopathology CNN | 50×50 RGB patches | ~92.80% |
| Watershed CNN | 224×224 segmented mammogram | ~99.67% |
| Canny CNN | 224×224 edge mammogram | ~97.82% |

*Accuracy on real datasets. Demo (synthetic) results will differ.*